[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/ab-test-practice/blob/main/assignments/week04_assignment_concurrent_tests.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 4 Assignment: Running Concurrent Tests

## FamilyNest - Managing Multiple Experiments at Once

In this assignment, you will practice handling the challenge of running multiple A/B tests simultaneously on the same page. You will analyze overlapping tests, check for interactions, and compare approaches for concurrent experimentation.

---
## Setup

Run the cell below to import the required libraries and helper functions.

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import math

In [ ]:
def get_ci(mean_treatment, mean_control, n_treatment, n_control, ci=0.95):
    sd = ((mean_treatment * (1 - mean_treatment)) / n_treatment + (mean_control * (1 - mean_control)) / n_control)**0.5
    lift = mean_treatment - mean_control
    val = stats.norm.isf((1 - ci) / 2)
    lwr_bnd = lift - val * sd
    upr_bnd = lift + val * sd
    return (lwr_bnd, upr_bnd)

In [ ]:
def calculate_results(df, metric):
    """Calculate A/B test results. Uses != 'control' for treatment to handle multi-arm tests."""
    if metric not in ('new_active_listing', 'new_booked_listing', 'new_cancelled_listing', 'bookings_4w'):
        raise Exception("Invalid metric")

    # Values and relative difference
    mean_control = df.loc[df['variant'] == "control", metric].mean()
    mean_treatment = df.loc[df['variant'] != "control", metric].mean()  # Adjusting to allow for any treatment name

    abs_diff = mean_treatment - mean_control
    rel_diff = (mean_treatment - mean_control) / mean_control

    # P-Value -- two-sided ttest, assumes normal distribution
    data_group1 = list(df.query('variant == "control"')[metric])
    data_group2 = list(df.query('variant != "control"')[metric])

    results = stats.ttest_ind(a=data_group1, b=data_group2, equal_var=True)

    # 95% confidence intervals
    [ci_low, ci_high] = get_ci(mean_treatment, mean_control, len(data_group2), len(data_group1), .95)
    ci_low = ci_low / mean_control
    ci_high = ci_high / mean_control

    return (rel_diff, abs_diff, results.pvalue, ci_low, ci_high)

In [ ]:
from scipy.stats import chi2_contingency

def ab_test_chi2(data_column):
    """
    Performs a Chi-squared test on A/B testing data to check assignment balance.

    :param data_column: Pandas Series with variant labels (e.g., 'control' and 'treatment1').
    :return: Tuple of (p_value, total_count, observed_counts)
    """
    if not isinstance(data_column, pd.Series):
        raise ValueError("The data_column must be a Pandas Series.")

    # Count the occurrences of each variant
    observed_counts = data_column.value_counts()

    # Assuming equal split for the expected counts
    total_count = observed_counts.sum()
    n_groups = len(observed_counts)
    expected_counts = pd.Series([total_count / n_groups] * n_groups, index=observed_counts.index)

    # Perform the Chi-squared test
    chi2, p_value, _, _ = chi2_contingency([observed_counts, expected_counts])

    return p_value, total_count, observed_counts

---
## Background

It is the final week of the project at **FamilyNest**! The engineering team and your PM **Max** have gotten into a great rhythm with experimentation. They now have multiple ideas they want to test, often on the same pages.

This week, you will tackle a common challenge in industry: **running multiple tests simultaneously on the same page**. When two experiments target the same user flow, their effects can interact in unexpected ways. You need to think carefully about how to structure, analyze, and interpret concurrent tests.

### The Situation

The team has two ideas for improving the **host onboarding page**:

| Test | Description | Target Metric |
|------|-------------|---------------|
| **Test A: Price Recommendation Algorithm** | Tweak the algorithm to suggest slightly lower prices to new hosts | NBL |
| **Test B: First Booking Discount Checkbox** | Hosts can opt-in to offering a 10% discount on their first booking | NBL |

Both changes target **New Booked Listings (NBL)** and both live on the same onboarding page. This creates a potential for **interaction effects** -- the combined effect of both changes may not equal the sum of their individual effects.

---
## Task 1: Two Changes, One Page - What Should We Do?

Before running any tests, Max asks you to think through the **testing strategy**. Both changes target NBL, both live on the same page, and both relate to pricing in some way (one adjusts price recommendations, the other adds a discount option).

### Task 1.a. Test Structure Options

What are the options for testing these two changes? Discuss the **pros and cons** of each approach:

1. **Sequential testing** -- Run Test A first, then Test B after
2. **Mutual exclusion (traffic splitting)** -- Split traffic so each user only sees one test
3. **Full factorial (all combinations)** -- Test all combinations: control, A only, B only, A+B
4. **Overlapping tests with interaction check** -- Run both tests independently on the same users, then check for interactions

For each approach, consider:
- **Speed**: How quickly can we get results?
- **Statistical power**: Do we have enough traffic?
- **Interaction detection**: Can we detect if the two changes interact?
- **Complexity**: How hard is it to implement and analyze?

**Constraints:**
- Daily traffic to the onboarding page: **4,000 users**
- The team wants results within **4 weeks**
- Both changes could plausibly interact (both affect pricing)

<details>
<summary>💡 Hint</summary>

Think about what happens to per-variant sample size under each approach:
- Sequential: full traffic (4,000/day) but 2x the calendar time
- Mutual exclusion: traffic split in half (2,000/day per test)
- Full factorial: traffic split 4 ways (1,000/day per cell)
- Overlapping: full traffic for each test (4,000/day) but interaction risk

Also consider: since both changes relate to pricing, is interaction likely?

</details>

**Your answer:**

| Approach | Pros | Cons |
|----------|------|------|
| Sequential testing | | |
| Mutual exclusion | | |
| Full factorial | | |
| Overlapping + interaction check | | |

**My recommendation:**



---
## Task 2: Overlapping Tests - Run and Analyze

After discussion, the team decided to run **overlapping tests with an interaction check**. The reasoning: both tests get full traffic (faster results), and you will explicitly check for interactions after the fact.

Two separate experiments were run simultaneously on the same user population. Each user was independently randomized into each test:

In [ ]:
df_price_algo = pd.read_csv('../data/dataset_price_algo.csv')
df_checkbox_discount = pd.read_csv('../data/dataset_checkbox_discount.csv')

print("Price Algo test:")
print(df_price_algo.shape)
print(df_price_algo['variant'].value_counts())
print()
print("Checkbox Discount test:")
print(df_checkbox_discount.shape)
print(df_checkbox_discount['variant'].value_counts())

> **Note:** `df_price_algo` has variants "control" and "treatment1" (not "treatment"). The `calculate_results()` function handles this because it uses `!= "control"` for the treatment group.

### Task 2.a. Analyze Tests Individually

First, analyze each test independently as if the other test did not exist.

For **each** test:
1. Check assignment imbalance (chi-squared test)
2. Calculate results for the **target metric (NBL)**
3. Check the **guardrail metric (NCL)**

#### Test A: Price Recommendation Algorithm

In [ ]:
# Check assignment balance for the price algo test
# Your code here


In [ ]:
# Analyze target metric (NBL) for the price algo test
metric = 'new_booked_listing'

# Your code here


In [ ]:
# Analyze guardrail metric (NCL) for the price algo test
metric = 'new_cancelled_listing'

# Your code here


#### Test B: First Booking Discount Checkbox

In [ ]:
# Check assignment balance for the checkbox discount test
# Your code here


In [ ]:
# Analyze target metric (NBL) for the checkbox discount test
metric = 'new_booked_listing'

# Your code here


In [ ]:
# Analyze guardrail metric (NCL) for the checkbox discount test
metric = 'new_cancelled_listing'

# Your code here


**Summarize individual test results:**

| Test | NBL Relative Lift | NBL p-value | Significant? | NCL Concern? |
|------|-------------------|-------------|--------------|---------------|
| Price Algo | | | | |
| Checkbox Discount | | | | |

### Task 2.b. Check for Interactions

Since both tests ran simultaneously on the same user population, many users are in **both** tests. A user could be:
- Control in Test A **and** Control in Test B
- Treatment in Test A **and** Control in Test B
- Control in Test A **and** Treatment in Test B
- Treatment in Test A **and** Treatment in Test B

To check for interactions, you need to determine whether the effect of Test A **depends on** whether the user is also in treatment for Test B (and vice versa).

**Steps:**
1. Merge the two datasets on `id_user`
2. Segment by one test's assignment, then measure the other test's effect within each segment
3. Compare: Is the effect of Test A different when Test B is present vs. absent?

In [ ]:
# Merge the datasets on id_user
df_merged = df_price_algo[['id_user', 'variant']].merge(
    df_checkbox_discount[['id_user', 'variant', 'new_booked_listing', 'new_cancelled_listing']],
    on='id_user',
    suffixes=('_algo', '_checkbox')
)

print(f"Merged dataset shape: {df_merged.shape}")
print()
print("Cross-tabulation of assignments:")
print(pd.crosstab(df_merged['variant_algo'], df_merged['variant_checkbox']))

In [ ]:
# Check for interaction: Does the effect of the price algo test differ
# depending on whether the user is in treatment or control for the checkbox test?

# Step 1: Filter to users in CONTROL for checkbox test, then measure price algo effect
# Your code here


# Step 2: Filter to users in TREATMENT for checkbox test, then measure price algo effect
# Your code here


# Step 3: Compare the two effects - if they're very different, there's an interaction
# Your code here


<details>
<summary>💡 Hint</summary>

To check for interaction:
- Filter `df_merged` to users who are in **control** for the checkbox test --> compute the effect of the price algo (compare `variant_algo` control vs treatment1 on `new_booked_listing`)
- Filter `df_merged` to users who are in **treatment** for the checkbox test --> compute the effect of the price algo (same comparison)
- If these two effects are very different (e.g., one is positive and the other is negative, or magnitudes differ substantially), there is likely an interaction

You can create a temporary dataframe for each subset and use `calculate_results()` on it. You will need to rename columns so `variant_algo` becomes `variant` for the helper function to work.

</details>

### Task 2.c. Summary and Recommendation

Based on your individual test results AND your interaction analysis, what do you recommend to Max?

Consider:
- Should we ship both changes? Ship one? Ship neither?
- If there IS an interaction, how does that change your recommendation?
- If there is NO interaction, does that simplify the decision?

**Your answer:**

**Individual test conclusions:**
- Test A (Price Algo): 
- Test B (Checkbox Discount): 

**Interaction finding:**


**Recommendation to Max:**



---
## Task 3: Alternate Reality - Full Factorial (Multi-arm)

What if instead of running overlapping tests, you had done a **full factorial design**? In this approach, you would have created a single test with multiple arms:
- **Control**: No changes
- **Treatment**: Combined A+B treatment (both price algo update AND checkbox discount)

This is a simplified multi-arm approach where you test the combined effect rather than individual effects.

In [ ]:
df_price_multi = pd.read_csv('../data/dataset_price_multi.csv')

print(f"Shape: {df_price_multi.shape}")
print()
print(df_price_multi['variant'].value_counts())
print()
df_price_multi.head()

### Task 3.a. Duration Calculation

With a multi-arm test (3 variants: control, treatment_A, treatment_B), how long would it need to run?

**Parameters:**
- Daily traffic: **4,000 users** (split 3 ways --> ~1,333 per variant per day)
- Target metric: NBL, baseline 20%
- MDE: 5% relative lift
- Power: 80%
- Need **Bonferroni correction**: Since you are making 2 comparisons (each treatment vs. control), adjust alpha = 0.05 / 2 = 0.025 for each comparison

Calculate the required sample size per variant and the test duration.

<details>
<summary>💡 Hint</summary>

With 3 variants, daily traffic per variant is `daily_traffic / 3`.

Since you are making 2 comparisons (treatment_A vs control, treatment_B vs control), you need to adjust for multiple comparisons using Bonferroni: use `alpha = 0.025` instead of `0.05`.

The sample size formula is the same as before:
```
n = (Z_alpha/2 + Z_beta)^2 * (p1*(1-p1) + p2*(1-p2)) / (p2 - p1)^2
```
But use Z_alpha/2 for alpha=0.025 (which is ~2.24 instead of 1.96).

</details>

In [ ]:
# Calculate sample size and duration for a 3-arm test with Bonferroni correction

# Parameters
daily_traffic = 4000
n_variants = 3
baseline_rate = 0.20
mde_relative = 0.05
alpha = 0.05 / 2  # Bonferroni correction for 2 comparisons
power = 0.80

# Your code here
# Calculate sample size per variant


# Calculate duration in days


**Your answer:**

- Sample size per variant: 
- Duration: 
- How does this compare to the overlapping approach (where each test gets full traffic)?


### Task 3.b. Analysis and Recommendation

Now analyze the multi-arm test dataset:
1. Check assignment imbalance (use chi-squared -- note there may be more than 2 groups)
2. Compare control vs. each treatment variant on the target metric (NBL)
3. Check guardrail (NCL) for each comparison
4. Provide a summary and recommendation

<details>
<summary>💡 Hint</summary>

For assignment balance with 3 groups, you can still use `ab_test_chi2()` -- it has been set up to handle any number of groups by computing expected counts as `total / n_groups`.

For analyzing each treatment vs. control, you can filter the dataframe to just the two groups you want to compare (e.g., control + treatment_A), then use `calculate_results()`.

Remember to apply Bonferroni correction when interpreting p-values: with 2 comparisons, a result is significant only if p < 0.025.

</details>

In [ ]:
# Check assignment balance across all variants
# Your code here


In [ ]:
# Compare each treatment variant vs. control on NBL
metric = 'new_booked_listing'

# Your code here
# For each treatment variant, filter to control + that variant, then use calculate_results()


In [ ]:
# Check guardrail (NCL) for each comparison
metric = 'new_cancelled_listing'

# Your code here


**Your answer:**

**Assignment balance:**


**Results (remember Bonferroni: significant if p < 0.025):**

| Comparison | NBL Relative Lift | p-value | Significant? | NCL Concern? |
|------------|-------------------|---------|--------------|---------------|
| Treatment vs Control | | | | |

**Recommendation to Max:**



### Task 3.c. Comparing Approaches

Reflect on the two approaches you used in this assignment:
1. **Overlapping tests** (Task 2) -- two independent tests running simultaneously, with an interaction check
2. **Multi-arm test** (Task 3) -- a single test with multiple treatment arms

Based on your experience analyzing both:
- Which approach gave you more information?
- Which was faster (required less traffic/time)?
- When would you recommend each approach in the future?

**Your answer:**

